# Download Reference Transcripts for All OligoGym Targets

This notebook:
1. Loads all datasets available in OligoGym
2. Collects unique gene targets across all datasets
3. Downloads RefSeq RNA transcript FASTA files from NCBI via Entrez
4. Saves them as `{GENE}.fna` in `data/reference_transcripts/`

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Add project root to path
REPO_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(REPO_ROOT))

from oligogym.data import DatasetDownloader
from oligogym.download_genes import NCBIGeneDownloader

# Load NCBI credentials from .env
load_dotenv(REPO_ROOT / ".env")

NCBI_EMAIL = os.environ.get("NCBI_EMAIL")
NCBI_API_KEY = os.environ.get("NCBI_API_KEY")
OUTPUT_DIR = REPO_ROOT / "data" / "reference_transcripts"

print(f"Email: {NCBI_EMAIL}")
print(f"API key: {'provided' if NCBI_API_KEY else 'not set'}")
print(f"Output dir: {OUTPUT_DIR}")

Email: pedram@collagebio.com
API key: provided
Output dir: /Users/phosseini/Code/OligoGym/data/reference_transcripts


## 1. Load all datasets and collect targets

In [2]:
downloader = DatasetDownloader()
all_datasets = downloader.download("all", verbose=0)

# Collect per-dataset target info
dataset_info = []
for ds in all_datasets:
    targets = set(ds.targets) if ds.targets is not None else set()
    # Filter out non-gene entries
    targets = {t for t in targets if isinstance(t, str) and t not in ("negative_control", "Control", "undefined")}
    dataset_info.append({
        "dataset": ds.name,
        "modality": ds.modality,
        "n_samples": len(ds.data),
        "n_targets": len(targets),
        "targets": targets,
    })

df_info = pd.DataFrame(dataset_info).drop(columns="targets")
print(df_info.to_string(index=False))

            dataset modality  n_samples  n_targets
    Ichihara_2007_2    siRNA        419         11
     Alharbi_2020_2      ASO        192          4
Shmushkovich_2018_1    siRNA        356          0
       Hwang_2024_1      ASO      32602         17
    Hagedorn_2022_1      ASO       1825          2
     Alharbi_2020_1      ASO        192          4
    Ichihara_2007_1    siRNA       2431         30
   McQuisten_2007_1      ASO       3913         85
   Papargyri_2020_1      ASO        768          1
  Martinelli_2023_1    siRNA        907          0
     MOE_Neurotox_1      ASO       2437         13
       Knott_2014_1    shRNA     291551      17801


## 2. Unique targets across all datasets

We exclude the **Sherwood (Knott_2014_1)** dataset which contains 17,802 shRNA targets — too many for individual NCBI downloads. All other datasets combined have ~159 unique gene targets.

In [3]:
# Collect unique targets (excluding Sherwood)
all_targets = set()
for info in dataset_info:
    # excludes Sherwood's 17,802 shRNA targets, too many to download
    if info["dataset"] == "Knott_2014_1":
        continue
    all_targets.update(info["targets"])

all_targets = sorted(all_targets)
print(f"Total unique gene targets: {len(all_targets)}")
print()
for i in range(0, len(all_targets), 8):
    print("  ".join(f"{t:<16}" for t in all_targets[i:i+8]))

Total unique gene targets: 157

ABCB1             ABCC1             AGT               AKT1              AKT2              AKT3              ANGPTL2           APOL1           
APP               ATXN2             ATXN3             BIRC2             BIRC3             BIRC5             C9ORF72           CD44            
CDK6              CDKN2A            CDKN2B            CDKN2B-AS1        CHEK2             CHUK              CTNNB1            DGAT2           
DUSP8             DUSP9             E2F1              EGFR              EGR1              ERBB2             ESR1              ETS2            
FADD              FADS1             FANCA             FUS               GAPDH             GFAP              GNA11             GNA12           
GNA13             GNAI1             GNAI2             GNAI3             GNAS              HBV               HIF1A             HLA-DQA1        
HSD17B13          HTRA1             HTT               ICAM1             IGF2              IKBKB             IL

## 3. Check which transcripts are already downloaded

In [4]:
existing = {p.stem for p in OUTPUT_DIR.glob("*.fna")}
to_download = [t for t in all_targets if t not in existing]
already_have = [t for t in all_targets if t in existing]

print(f"Already downloaded: {len(already_have)}")
print(f"To download:       {len(to_download)}")
print()
if to_download:
    print("Missing targets:")
    for i in range(0, len(to_download), 8):
        print("  ".join(f"{t:<16}" for t in to_download[i:i+8]))

Already downloaded: 156
To download:       1

Missing targets:
HBV             


## 4. Download missing transcripts

In [5]:
if not to_download:
    print("All transcripts already downloaded!")
else:
    gene_downloader = NCBIGeneDownloader(
        email=NCBI_EMAIL,
        output_dir=str(OUTPUT_DIR),
        api_key=NCBI_API_KEY,
    )

    results = gene_downloader.process_genes(
        gene_symbols=to_download,
        taxon="human",
        delay=0.4,
    )

    gene_downloader.summarize_results(results)


Processing: HBV
  Could not resolve gene symbol: HBV


DOWNLOAD SUMMARY

Successful: 0/1

Failed: 1/1
  - HBV


## 5. Summary

In [ ]:
# Final check
final_files = sorted(OUTPUT_DIR.glob("*.fna"))
covered = {p.stem for p in final_files}
still_missing = [t for t in all_targets if t not in covered]

print(f"Total .fna files in {OUTPUT_DIR.name}/: {len(final_files)}")
print(f"Targets covered: {len(covered)}/{len(all_targets)}")

if still_missing:
    print(f"\nStill missing ({len(still_missing)}):")
    for t in still_missing:
        print(f"  - {t}")
else:
    print("\nAll targets covered!")